<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_64_mcp_model_context_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🔌 **Module 64 — MCP (Model Context Protocol)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 64 — MCP

> Every AI app eventually wants the same five things: read files, query Postgres, hit the GitHub API, search Slack, control a browser. M32–M35's agents wired each one bespoke. **Model Context Protocol (MCP)** — Anthropic's open standard from late 2024 — replaces that with **one wire protocol** every assistant and every tool source can speak. Build your data source as an MCP **server** once; Claude Desktop, Claude Code, Cursor, Zed, VS Code Copilot, and many others can use it instantly.
>
> Think "USB-C for AI tools."

### What you'll cover
1. The problem MCP solves
2. The three primitives — **tools, resources, prompts**
3. Architecture — clients, servers, hosts, transports
4. Setup — install the Python SDK
5. **Build an MCP server in 30 lines** with `FastMCP`
6. The wire format — what actually flows over JSON-RPC
7. **Resources** — pulling data into context (not tools!)
8. **Prompts** — re-usable templated workflows
9. Wire the server to a host (Claude Desktop / Claude Code)
10. MCP vs OpenAI function-calling vs LangChain tools — when each wins


## 1 · The problem

Before MCP, every AI assistant connected to every tool *independently*:

```
   Claude    ←→ {github-tools, fs-tools, slack-tools, gh-pr-tools, …}
   ChatGPT   ←→ {github-tools, fs-tools, slack-tools, …}   ← all rewritten
   Cursor    ←→ {github-tools, fs-tools, slack-tools, …}   ← rewritten again
   Custom    ←→ {github-tools, fs-tools, slack-tools, …}   ← and again
```

`N` assistants × `M` tool sources = `N × M` integrations. Each one has its own auth, retry, schema, error format.

**MCP flips it to N + M:**

```
   Claude / ChatGPT / Cursor / Zed / Copilot ──┐
                                                │  MCP protocol (JSON-RPC)
                            ┌───────────────────┴──────────────┐
                            ▼                                   ▼
                       GitHub MCP server                 Postgres MCP server
                       Slack MCP server                  Filesystem MCP server
                       Linear MCP server                 Stripe MCP server
                       Browser MCP server                ...
```

One protocol; many hosts; many servers. Write a server once; every host that speaks MCP can use it.

## 2 · The three primitives

MCP servers expose **three** kinds of things:

| Primitive | Initiated by | Used for |
|---|---|---|
| **Tool** | The model | actions with side-effects — "create a GitHub issue," "execute SQL" |
| **Resource** | The host (you, or the model with permission) | read-only data the model should *see* — "the contents of `/repo/README.md`" |
| **Prompt** | The user / host | re-usable templated workflows — "draft a release-notes PR for tag v1.2.3" |

This split matters: **tools are for actions, resources are for context, prompts are for workflows.** Most teams over-use tools (everything looks like a hammer) and skip resources entirely. We'll fix that habit here.

## 3 · Architecture

```
┌────────────────────────────────┐       ┌────────────────────────────┐
│           HOST                  │       │       MCP SERVER           │
│  (Claude Desktop, VS Code,      │       │  Your code that exposes    │
│   Cursor, Zed, your own app)    │       │  tools / resources /       │
│                                 │       │  prompts                   │
│   ┌──────────────────────┐      │       │                            │
│   │      CLIENT          │ ◄─── │ JSON-RPC over stdio / HTTP+SSE ──► │
│   │  one per server      │      │       │                            │
│   └──────────────────────┘      │       └────────────────────────────┘
└────────────────────────────────┘
```

- **Host** = the LLM application. *It* owns the model and decides what to call.
- **Client** = the connection inside the host to *one* server.
- **Server** = your code, written using the MCP SDK.
- **Transport**:
  - **stdio** — server runs as a subprocess of the host (most common; what Claude Desktop uses).
  - **Streamable HTTP** (2025) — remote servers with bidirectional streaming over plain HTTP. The new default for hosted services.
  - **SSE** — legacy transport, being phased out.


## 4 · Setup

In [ ]:
!pip -q install mcp 'mcp[cli]'

The official Python SDK is **`mcp`**. It includes:
- `mcp.server.fastmcp.FastMCP` — high-level decorator API for building servers.
- `mcp.client.session` — client-side primitives if you want to *talk to* a server (we'll do that too).
- `mcp` CLI — `mcp dev your_server.py` to debug locally with the Inspector UI.

## 5 · Build an MCP server in 30 lines

In [ ]:
# write a small server file we can launch later
server_code = '''
"""weather_server.py — a toy MCP server."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather")

@mcp.tool()
def get_forecast(city: str, units: str = "c") -> dict:
    """Get the (faked) weather forecast for a city.

    Args:
        city: city name, e.g. "Paris"
        units: "c" or "f"
    """
    fake = {"Paris": 21, "Tokyo": 27, "Seattle": 14, "Chennai": 33}
    temp = fake.get(city, 20)
    if units == "f":
        temp = round(temp * 9/5 + 32)
    return {"city": city, "temp": temp, "units": units, "condition": "sunny"}

@mcp.resource("note://latest")
def latest_note() -> str:
    """Read the most recently saved note."""
    return "Remember: ship MCP server for the support team by Friday."

@mcp.prompt()
def trip_plan(destination: str) -> str:
    """Generate a request to plan a one-day trip."""
    return (
        f"Plan a one-day itinerary for {destination}. "
        "Use the get_forecast tool first to check the weather, "
        "then suggest morning, afternoon and evening activities."
    )

if __name__ == "__main__":
    mcp.run()                  # default: stdio transport
'''
import pathlib
pathlib.Path("/content/weather_server.py").write_text(server_code)
print("server written")

**Read what you wrote.**
- `@mcp.tool()` — exposes `get_forecast` as a model-callable tool. The **type hints** become the JSON schema; the **docstring** becomes the description the model sees.
- `@mcp.resource("note://latest")` — exposes a read-only resource. The URI scheme is yours to design.
- `@mcp.prompt()` — exposes a parameterised prompt the *user* can pick from a menu.
- `mcp.run()` defaults to stdio. Change to `mcp.run(transport="streamable-http", port=8000)` for HTTP.

**That's a real, working MCP server.** Add `command: python /content/weather_server.py` to a host's MCP config and Claude Desktop / Cursor / Zed will discover and call all three primitives.

## 6 · The wire format — what actually flows

MCP is **JSON-RPC 2.0** under the hood. Here's a real session:

```jsonc
// Host → server (first thing always)
→ { "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": { "protocolVersion": "2025-06-18",
                "capabilities": {...},
                "clientInfo": {"name": "Claude Desktop", "version": "1.0"} } }

// Server → host
← { "jsonrpc": "2.0", "id": 1, "result":
    { "capabilities": { "tools": {}, "resources": {}, "prompts": {} },
      "serverInfo": {"name": "Weather"} } }

// Host discovers
→ { "jsonrpc": "2.0", "id": 2, "method": "tools/list" }
← { "jsonrpc": "2.0", "id": 2, "result": { "tools": [
    {"name": "get_forecast",
     "description": "Get the (faked) weather forecast for a city.",
     "inputSchema": { "type": "object",
        "properties": { "city": {"type": "string"},
                        "units": {"type": "string", "default": "c"} },
        "required": ["city"] } } ] } }

// Model decides to call the tool
→ { "jsonrpc": "2.0", "id": 3, "method": "tools/call",
    "params": { "name": "get_forecast",
                "arguments": { "city": "Paris", "units": "c" } } }
← { "jsonrpc": "2.0", "id": 3, "result":
    { "content": [{ "type": "text",
                    "text": "{\"city\": \"Paris\", \"temp\": 21, ...}" }] } }
```

You don't write any of this by hand — the SDK does. But **knowing the wire shape** matters when debugging a misbehaving server: run with `mcp dev server.py` and watch the actual JSON flowing.

## 7 · Resources — pulling data into context

A common mistake is shoving everything through tools. Tools should be for *side-effects*. For read-only data, use **resources**:

In [ ]:
resource_examples = '''
# Static resource — fixed URI
@mcp.resource("config://app")
def app_config() -> str:
    return open("/etc/app/config.yaml").read()

# Templated resource — URI placeholders captured as args
@mcp.resource("file:///{path}")
def read_file(path: str) -> str:
    """Read a local file (sandboxed)."""
    safe_root = "/content/docs/"
    full = safe_root + path
    if not full.startswith(safe_root): raise ValueError("path escape!")
    return open(full).read()

# Binary resource — returns bytes
from mcp.server.fastmcp.resources import binary
@mcp.resource("image://logo")
def logo() -> bytes:
    return open("/content/logo.png", "rb").read()
'''
print(resource_examples)

**Why resources beat tools for reading data.**
- The host can **stream** them into the model's context without an extra round trip.
- The user can pick which resources to attach (in Claude Desktop: the paperclip menu).
- Resources support **subscriptions** — the server can push an `updated` notification when the file changes.
- The model never has to "decide" to fetch them — the host or user does.

Rule: **side-effects → tool. Read-only data → resource.**

## 8 · Prompts — reusable workflows

In [ ]:
prompt_examples = '''
@mcp.prompt()
def code_review(language: str, code: str) -> str:
    """Review a code snippet."""
    return (
        f"You are a senior {language} engineer. Review the following code for "
        "correctness, security, readability, and idiomatic style. List issues, "
        "then propose minimal fixes.\n\n```{language}\n{code}\n```"
    )

@mcp.prompt()
def pr_release_notes(branch: str, tag: str) -> str:
    return (
        f"Compare git diff between {tag} and {branch} (you can use the git tools). "
        "Group changes into Features / Fixes / Breaking / Docs. "
        "Write release notes in markdown.\n"
    )
'''
print(prompt_examples)

Prompts show up in the host's UI as commands (`/code-review`, `/pr-release-notes`). Users pick one, fill in arguments, and a fully-formed prompt is inserted into the chat. Great for **codifying playbooks** so they live in code, not in someone's head.

## 9 · Wire it to a host

### Claude Desktop / Claude Code

Edit `~/.config/Claude/claude_desktop_config.json` (or use **`mcp install`**):

```jsonc
{
  "mcpServers": {
    "weather": {
      "command": "python",
      "args": ["/abs/path/to/weather_server.py"]
    },
    "github": {
      "command": "npx",
      "args": ["-y", "@modelcontextprotocol/server-github"],
      "env": { "GITHUB_PERSONAL_ACCESS_TOKEN": "ghp_..." }
    },
    "remote-prod": {
      "url": "https://my-mcp.example.com/sse",
      "auth": { "type": "bearer", "token": "..." }
    }
  }
}
```

Restart the host; the new tools/resources/prompts appear in the UI.

### Inspect from the command line

In [ ]:
# `mcp dev` launches your server with the Inspector — a web UI that lets you call tools,
# read resources, and watch the JSON-RPC traffic.
#
#   mcp dev /content/weather_server.py
#
# Or talk to it as a client from Python:
client_code = '''
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    server = StdioServerParameters(command="python", args=["/content/weather_server.py"])
    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as sess:
            await sess.initialize()
            print("tools:", [t.name for t in (await sess.list_tools()).tools])
            r = await sess.call_tool("get_forecast", {"city": "Tokyo", "units": "f"})
            print("result:", r.content)

asyncio.run(main())
'''
print(client_code)

## 10 · MCP vs OpenAI function-calling vs LangChain tools

| | OpenAI function-calling | LangChain `@tool` | MCP |
|---|---|---|---|
| Defined in | client app | client app | **separate process** |
| Reusable across hosts? | each provider's format | Python-only | **any MCP host** |
| Discovers at runtime? | ✗ (compiled in) | ✗ | ✓ (`tools/list`) |
| Auth lives with | the client | the client | the **server** (closer to the data) |
| Streaming tool output | mostly no | no | yes (over SSE / Streamable HTTP) |
| Resources / prompts | no equivalent | no equivalent | first-class |
| Best for | one app, one API | Python agent prototypes | **shared org-wide tooling** |

**When to pick MCP**
- You want one Postgres / Slack / GitHub integration that **every assistant in your company can use**.
- You're shipping a SaaS that wants to expose itself to AI tools (Stripe, Linear, Notion all do this).
- You want clean separation between *who has the model* and *who owns the data*.

**When NOT to pick MCP**
- One-off Python agent for one app → LangChain tools are simpler.
- Latency-critical hot path inside your own LLM stack → in-process function calls.
- The "tool" is a pure Python helper with no side effects or auth.

### The 2025+ MCP ecosystem (illustrative)

Hosts that speak MCP: **Claude Desktop · Claude Code · Cursor · Zed · VS Code Copilot · Continue · Cline · Windsurf · Cody · LibreChat · OpenAI Agents SDK** (added MCP support).

Official + popular servers: **GitHub · GitLab · Filesystem · Postgres · SQLite · Memory · Brave Search · Fetch · Puppeteer · Slack · Google Drive · Sentry · Stripe · Linear · Notion · Cloudflare · Atlassian**.

`https://github.com/modelcontextprotocol/servers` is the canonical index.

## ✅ Recap
- MCP is a JSON-RPC standard that turns `N × M` AI×tool integrations into `N + M`.
- Three primitives: **tools** (actions), **resources** (data), **prompts** (workflows).
- Build a server in ~30 lines with `FastMCP`; the SDK handles the wire protocol.
- Wire it to Claude Desktop / Cursor / VS Code via one JSON entry.
- Transport: **stdio** for local, **Streamable HTTP** for hosted.
- Pick MCP when tools should outlive your app and serve every host in the org.

Next: **M65 — Multimodal (VLM + Whisper/TTS)**.
